# Build `index.faiss` fast on a single CUDA GPU (e.g. B200)

One-shot, max-speed index build for the Anglican library. Embeds every chunk in
big GPU batches (flash attention, fp16), builds the HNSW graph once across all
CPU cores, and writes `index.faiss` a single time. No per-window writes, no DB
bookkeeping — you only ship `index.faiss` back to the serving box (it already
has the DB).

**Before running:** upload your repaired DB to this pod as `library.db` (next to
this notebook), e.g. `gzip -k library-clean.db` on the NAS, `scp` it up, `gunzip`.

The index uses chunk ids as vector ids and 1024-d inner-product vectors, so it
drops straight into the serving box with no changes.

In [ ]:
# 0. Allocator config (set BEFORE torch is imported anywhere) + deps.
#    expandable_segments reduces fragmentation OOMs.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Most GPU pods ship torch+CUDA; install the two extras if missing.
# If torch can't see the B200 (very new silicon), install a cu128 nightly.
import importlib, subprocess, sys
for pkg, imp in [("sentence-transformers", "sentence_transformers"), ("faiss-cpu", "faiss")]:
    if importlib.util.find_spec(imp) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
print("deps ready")

In [ ]:
# 1. Confirm the GPU runs a real kernel (is_available() can lie on new arches).
import torch, faiss, numpy as np, sqlite3, os, time
assert torch.cuda.is_available(), "No CUDA GPU — wrong pod/torch build."
print("torch", torch.__version__, "| faiss", faiss.__version__)
print("device:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), "GiB")
a = torch.randn(4096, 4096, device="cuda", dtype=torch.float16)
torch.cuda.synchronize(); _ = (a @ a).sum().item(); torch.cuda.synchronize()
print("GPU matmul OK")
torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
# 2. Config — these MUST match the serving box (anglican_search/config.py).
DB_PATH        = "library.db"      # the uploaded repaired DB
INDEX_OUT      = "index.faiss"
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
PASSAGE_PREFIX  = ""               # Qwen3 passages are embedded raw
TRUNCATE_DIM    = None            # full native dim
EMBEDDING_DIM   = 1024
HNSW_M             = 32
HNSW_EF_CONSTRUCTION = 200        # graph quality; lower (e.g. 128) builds faster
HNSW_EF_SEARCH       = 256

# We cap sequences at 512 tokens (cell 3) so SDPA's batch*heads*seq^2 attention
# stays bounded — that lets us use a big batch that actually saturates the GPU.
# ~5% of chunks exceed 512 tokens and get their tail truncated for the vector
# (FTS still indexes their full text on the serving box).
MAX_SEQ_LEN  = 512
ENCODE_BATCH = 1024               # big batch keeps a fast GPU fed; drop if you OOM
faiss.omp_set_num_threads(os.cpu_count())   # use all cores for the HNSW build
print("faiss threads:", os.cpu_count())

In [ ]:
# 3. Load the embedder on the GPU in fp16 (flash attention is the CUDA default).
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(EMBEDDING_MODEL, device="cuda", truncate_dim=TRUNCATE_DIM)
model = model.half()
model.max_seq_length = MAX_SEQ_LEN   # bound attention memory so a big batch fits
print("loaded", EMBEDDING_MODEL, "| max_seq_length", model.max_seq_length)

In [ ]:
# 4. Pull every chunk (id + text). id becomes the FAISS vector id, so the index
#    maps 1:1 onto the serving box's chunks table.
conn = sqlite3.connect(DB_PATH)
rows = conn.execute("SELECT id, text FROM chunks ORDER BY id").fetchall()
conn.close()
ids   = np.fromiter((r[0] for r in rows), dtype=np.int64, count=len(rows))
texts = [PASSAGE_PREFIX + r[1] for r in rows]
print(f"{len(texts):,} chunks loaded")

In [ ]:
# 5. Embed everything in one streamed GPU pass. sentence-transformers sorts by
#    length internally (minimal padding) and restores order, so emb[i] <-> ids[i].
t0 = time.time()
emb = model.encode(
    texts, batch_size=ENCODE_BATCH, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=True,
).astype("float32")
print(f"embedded {emb.shape} in {(time.time()-t0)/60:.1f} min")

In [ ]:
# 6. Build the HNSW index once across all cores (this is the CPU-bound step).
t0 = time.time()
base = faiss.IndexHNSWFlat(EMBEDDING_DIM, HNSW_M, faiss.METRIC_INNER_PRODUCT)
base.hnsw.efConstruction = HNSW_EF_CONSTRUCTION
base.hnsw.efSearch = HNSW_EF_SEARCH
index = faiss.IndexIDMap2(base)
index.add_with_ids(emb, ids)
print(f"HNSW built: {index.ntotal:,} vectors in {(time.time()-t0)/60:.1f} min")

In [ ]:
# 7. Write the index once.
faiss.write_index(index, INDEX_OUT)
print(f"wrote {INDEX_OUT}: {os.path.getsize(INDEX_OUT)/1e9:.2f} GB")

In [ ]:
# 8. Sanity check: a real query should return on-topic chunk ids before you ship.
QUERY_PREFIX = ("Instruct: Given a question about Christian theology, retrieve "
                "passages from historical Anglican texts that answer it\nQuery:")
qv = model.encode([QUERY_PREFIX + "the eternal generation of the Son"],
                  normalize_embeddings=True, convert_to_numpy=True).astype("float32")
base.hnsw.efSearch = 256
D, I = index.search(qv, 5)
print("top chunk ids:", I[0].tolist())
print("scores:", [round(float(s), 3) for s in D[0]])

## Ship it

Download `index.faiss` and drop it on the serving box:

```bash
# from the NAS:
scp USER@POD:/path/index.faiss \
    ~/Docker/rag/anglican-search-starter/deploy/docker/home/data/index.faiss
```

Then start the app on the Arc box (`docker compose up -d`). The query embedder on
the Arc and the passage embedder here share the same Qwen3 weights, so the index
is fully compatible — eager vs flash attention differ only in kernel, not in the
numbers cosine search depends on.

**Speed/memory knobs:**
- `ENCODE_BATCH` trades GPU memory for throughput. SDPA materializes
  `batch*heads*seq^2` (a padding mask blocks the memory-light flash path), and ~5%
  of chunks reach ~1400 tokens, so 256 is the safe default here. If you hit
  `CUDA out of memory`, halve it; the GPU isn't the bottleneck so it costs little.
- `HNSW_EF_CONSTRUCTION = 128` roughly halves the (CPU-bound) build step, slight
  recall cost. This is the real lever for total time on a 24-vCPU box.